# Step 4: XGBoost Dual-Objective Matcher
Trains two XGBoost models:
- **Regressor**: predicts detour distance (km)
- **Classifier**: predicts acceptance probability

Includes grid search hyperparameter tuning, 5-fold cross-validation, and saves feature importance + ROC curve plots.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Enable inline plots for notebook display
%matplotlib inline

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, accuracy_score,
    classification_report, f1_score, roc_auc_score, roc_curve
)

class MultiObjectiveMatcher:
    """
    Dual-objective XGBoost model for P2P ride matching.
    Trains a regressor (detour prediction) and classifier (accept/reject)
    on the same features, then combines their outputs into a match score.
    """
    def __init__(self):
        self.regressor = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
        self.classifier = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=5, eval_metric='logloss')
        self.features = ['driver_orig_dist', 'overlap_ratio', 'time_diff_mins', 'driver_approx_fare', 'rider_approx_fare', 'fare_ratio', 'pickup_hour', 'pickup_dow']
        
    def prepare_data(self, df):
        return df[self.features], df['detour_distance_km'], df['label']

    def train(self, df):
        """Driver-level split, grid search, 5-fold CV, scale_pos_weight, ablation study."""
        X, y_reg, y_clf = self.prepare_data(df)
        print(f"Training on {len(X)} samples with {len(self.features)} features.")

        unique_drivers = df['driver_id'].unique()
        train_drivers, _ = train_test_split(unique_drivers, test_size=0.2, random_state=42)
        train_mask = df['driver_id'].isin(train_drivers)
        train_df = df[train_mask].reset_index(drop=True)
        test_df  = df[~train_mask].reset_index(drop=True)
        X_train, X_test = train_df[self.features], test_df[self.features]
        yr_train, yr_test = train_df['detour_distance_km'], test_df['detour_distance_km']
        yc_train, yc_test = train_df['label'], test_df['label']
        print(f"Driver-level split: {len(train_df)} train, {len(test_df)} test. No overlap: {len(set(train_df['driver_id']) & set(test_df['driver_id'])) == 0}")

        print("\nGrid search...")
        grid_search = GridSearchCV(xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss'),
            {'n_estimators': [50,100,200], 'max_depth': [3,5,7], 'learning_rate': [0.05,0.1,0.2]},
            cv=5, scoring='accuracy', n_jobs=-1)
        grid_search.fit(X_train, yc_train)
        best_params = grid_search.best_params_
        print(f"Best params: {best_params}")

        pos_count, neg_count = (yc_train==1).sum(), (yc_train==0).sum()
        scale_pos_weight = neg_count / max(pos_count, 1)
        print(f"Class balance - pos: {pos_count}, neg: {neg_count}, scale_pos_weight: {scale_pos_weight:.3f}")

        self.classifier = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=scale_pos_weight, **best_params)
        self.regressor = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=best_params['n_estimators'], max_depth=best_params['max_depth'], learning_rate=best_params['learning_rate'])

        print("\n5-fold CV...")
        cv_clf = cross_val_score(self.classifier, X_train, yc_train, cv=5, scoring='accuracy')
        cv_reg = cross_val_score(self.regressor, X_train, yr_train, cv=5, scoring='neg_mean_absolute_error')
        print(f"Classifier CV: {cv_clf.mean():.4f} (+/- {cv_clf.std():.4f}) | Folds: {np.round(cv_clf,4)}")
        print(f"Regressor CV MAE: {-cv_reg.mean():.4f} (+/- {cv_reg.std():.4f}) | Folds: {np.round(-cv_reg,4)}")

        print("\nFinal training...")
        self.regressor.fit(X_train, yr_train)
        pred_detour = self.regressor.predict(X_test)
        print(f"MAE: {mean_absolute_error(yr_test, pred_detour):.2f} | RMSE: {np.sqrt(mean_squared_error(yr_test, pred_detour)):.4f}")

        self.classifier.fit(X_train, yc_train)
        pred_match = self.classifier.predict(X_test)
        pred_proba = self.classifier.predict_proba(X_test)[:, 1]
        acc = accuracy_score(yc_test, pred_match)
        print(f"Accuracy: {acc:.2f} | F1: {f1_score(yc_test, pred_match, zero_division=0):.4f} | ROC-AUC: {roc_auc_score(yc_test, pred_proba):.4f}")
        print(classification_report(yc_test, pred_match, zero_division=0))

        os.makedirs("figures", exist_ok=True)

        fig, ax = plt.subplots(figsize=(8, 5))
        xgb.plot_importance(self.classifier, ax=ax, importance_type='gain')
        ax.set_title("Feature Importance (Classifier - Gain)")
        fig.tight_layout(); fig.savefig("figures/feature_importance.png", dpi=150); plt.show(); plt.close(fig)
        print("Saved figures/feature_importance.png")

        fpr, tpr, _ = roc_curve(yc_test, pred_proba)
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f"XGBoost (AUC = {roc_auc_score(yc_test, pred_proba):.3f})")
        ax.plot([0, 1], [0, 1], 'k--', label="Random")
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_title("ROC Curve"); ax.legend()
        fig.tight_layout(); fig.savefig("figures/roc_curve.png", dpi=150); plt.show(); plt.close(fig)
        print("Saved figures/roc_curve.png")

        # Feature ablation study
        print("\nFeature ablation study...")
        ablation_results = {}
        for feat in self.features:
            ablation_feats = [f for f in self.features if f != feat]
            clf_abl = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=scale_pos_weight, **best_params)
            clf_abl.fit(X_train[ablation_feats], yc_train)
            ablation_results[feat] = round(acc - accuracy_score(yc_test, clf_abl.predict(X_test[ablation_feats])), 4)

        print("Accuracy drop when removed:")
        for feat, drop in sorted(ablation_results.items(), key=lambda x: -x[1]):
            print(f"  {feat}: {drop:+.4f}")

        fig, ax = plt.subplots(figsize=(8, 5))
        feats_s = sorted(ablation_results, key=lambda x: ablation_results[x])
        ax.barh(feats_s, [ablation_results[f] for f in feats_s], color='steelblue')
        ax.set_xlabel("Accuracy drop when removed"); ax.set_title("Feature Ablation Study"); ax.axvline(0, color='black', linewidth=0.8)
        fig.tight_layout(); fig.savefig("figures/ablation_study.png", dpi=150); plt.show(); plt.close(fig)
        print("Saved figures/ablation_study.png")

        return X_test, yr_test, yc_test, pred_detour, pred_match

    def compute_match_score(self, feature_row):
        """Combine detour prediction and acceptance probability into a single match score."""
        df_feats = pd.DataFrame([feature_row], columns=self.features)
        return self.classifier.predict_proba(df_feats)[0][1] - (self.regressor.predict(df_feats)[0] * 0.1)

In [3]:
# Run the XGBoost dual-objective matcher
print("Loading simulated candidate pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
matcher = MultiObjectiveMatcher()
X_test, yr_test, yc_test, pred_detour, pred_match = matcher.train(df)

Loading simulated candidate pairs...
Training on 750 samples with 8 features.
Driver-level split: 592 train, 158 test. No overlap: True

Grid search...


Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
Class balance - pos: 252, neg: 340, scale_pos_weight: 1.349

5-fold CV...


Classifier CV: 0.9276 (+/- 0.1074) | Folds: [0.7143 0.9916 0.9915 0.9576 0.9831]
Regressor CV MAE: 3.0661 (+/- 1.6555) | Folds: [1.3929 0.9964 3.1871 4.7239 5.03  ]

Final training...
MAE: 2.76 | RMSE: 5.1964
Accuracy: 0.91 | F1: 0.9068 | ROC-AUC: 0.9727
              precision    recall  f1-score   support

           0       1.00      0.82      0.90        85
           1       0.83      1.00      0.91        73

    accuracy                           0.91       158
   macro avg       0.91      0.91      0.91       158
weighted avg       0.92      0.91      0.90       158

Saved figures/feature_importance.png
Saved figures/roc_curve.png

Feature ablation study...


/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_84802/3023794312.py:81: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig("figures/feature_importance.png", dpi=150); plt.show(); plt.close(fig)
/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_84802/3023794312.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig("figures/roc_curve.png", dpi=150); plt.show(); plt.close(fig)


Accuracy drop when removed:
  time_diff_mins: +0.0506
  overlap_ratio: +0.0063
  fare_ratio: +0.0063
  driver_orig_dist: +0.0000
  driver_approx_fare: +0.0000
  rider_approx_fare: +0.0000
  pickup_hour: +0.0000
  pickup_dow: +0.0000
Saved figures/ablation_study.png


/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_84802/3023794312.py:109: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig("figures/ablation_study.png", dpi=150); plt.show(); plt.close(fig)
